# AI Path: From Zero to Production AI

A single interactive notebook that takes you from basic prompt engineering to a complete AI application.

You'll incrementally build **AskDoc** — an AI-powered document assistant with RAG, tools, memory, agents, and safety guardrails. Each module adds a capability; by the end you have a production-ready system.

*Frameworks and libraries used here are just to get you familiar with the concepts and techniques and not meant as recommendation for using the exact same setup for your upcoming projects.*


**Stack**: [LangChain](https://docs.langchain.com/oss/python/langchain/overview) · [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) · [ChromaDB](https://docs.trychroma.com/) · [Pydantic](https://docs.pydantic.dev/)

---

## Module 00: Environment Setup

We use a **provider factory pattern** — one function that returns the right LLM/embedding model regardless of whether you're using OpenAI, Azure, or Gemini. This lets all subsequent code be provider-agnostic.

📖 Docs:
- [LangChain Chat Models](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain Embedding Models](https://docs.langchain.com/oss/python/integrations/embeddings)
- [python-dotenv](https://saurabh-kumar.com/python-dotenv/)

In [ ]:
import importlib

packages = [
    "langchain", "langchain_openai", "langchain_google_genai",
    "langgraph", "chromadb", "tiktoken",
    "pydantic", "dotenv", "rich"
]

print("Checking dependencies...")
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "ok")
        print(f"  ✅ {pkg} ({version})")
    except ImportError:
        print(f"  ❌ {pkg} — NOT FOUND")

print("\nIf anything shows ❌, run: uv sync")


### Configure Your LLM Provider

Edit `.env` in the project root. You only need **one** provider:

| Provider | Required Env Vars |
|----------|------------------|
| OpenAI | `OPENAI_API_KEY` |
| Azure OpenAI | `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT` |
| Google Gemini | `GOOGLE_API_KEY` |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

provider = None
if os.getenv("AZURE_OPENAI_API_KEY"):
    provider = "azure"
elif os.getenv("OPENAI_API_KEY"):
    provider = "openai"
elif os.getenv("GOOGLE_API_KEY"):
    provider = "gemini"

if provider:
    print(f"✅ Detected provider: {provider}")
else:
    print("❌ No API keys found. Edit ../.env")

### 🎯 Exercise: Build the Provider Factory

Complete `get_llm()` and `get_embeddings()` — these power everything that follows.

In [ ]:
from langchain_openai import ChatOpenAI, AzureChatOpenAI, OpenAIEmbeddings, AzureOpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


def get_llm(temperature: float = 0.0):
    """Return a chat model for the configured provider."""
    if provider == "openai":
        # TODO: return ChatOpenAI(model="gpt-4o", temperature=temperature)
        pass
    elif provider == "azure":
        # TODO: return AzureChatOpenAI(...)
        pass
    elif provider == "gemini":
        # TODO: return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=temperature)
        pass
    else:
        raise ValueError("No provider configured.")


def get_embeddings():
    """Return an embedding model for the configured provider."""
    if provider == "openai":
        # TODO: return OpenAIEmbeddings(model="text-embedding-3-small")
        pass
    elif provider == "azure":
        # TODO: return AzureOpenAIEmbeddings(...)
        pass
    elif provider == "gemini":
        # TODO: return GoogleGenerativeAIEmbeddings(model="models/embedding-001")
        pass
    else:
        raise ValueError("No provider configured.")

<details>
<summary>💡 Solution</summary>

```python
def get_llm(temperature: float = 0.0):
    if provider == "openai":
        return ChatOpenAI(model="gpt-4o", temperature=temperature)
    elif provider == "azure":
        return AzureChatOpenAI(
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
            temperature=temperature,
        )
    elif provider == "gemini":
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=temperature)
    else:
        raise ValueError("No provider configured.")


def get_embeddings():
    if provider == "openai":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif provider == "azure":
        return AzureOpenAIEmbeddings(
            azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        )
    elif provider == "gemini":
        return GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    else:
        raise ValueError("No provider configured.")
```
</details>

In [ ]:
# Test your setup
llm = get_llm()
response = llm.invoke("Say 'Hello, AI Path!' and nothing else.")
print(f"LLM: {response.content}")

embeddings = get_embeddings()
vector = embeddings.embed_query("test")
print(f"Embedding dimensions: {len(vector)}")
print("✅ Setup complete!")

---
## Module 01: Prompt Engineering

The **prompt** is your primary interface to the model. Small changes in wording produce dramatically different outputs. This module covers the core patterns:

- **System vs User messages** — the system message sets behavior; the user message provides the task
- **Instruction design** — specific, constrained prompts beat vague ones every time
- **Few-shot prompting** — teach by example instead of describing
- **Templates & Chains** — reusable prompts with variables, piped to models via LangChain's `|` operator

📖 Docs:
- [LangChain Messages](https://docs.langchain.com/oss/python/langchain/messages)
- [Prompt Templates](https://docs.langchain.com/oss/python/langchain/models)
- [LCEL (LangChain Expression Language)](https://docs.langchain.com/oss/python/langchain/overview)
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# System prompt controls behavior
response = llm.invoke([
    SystemMessage(content="You are a sarcastic pirate. Answer in 1-2 sentences max."),
    HumanMessage(content="What is Python?")
])
print(response.content)

### 🎯 Exercise: Write a Code Reviewer Prompt

Make the LLM act as a senior code reviewer that focuses on bugs and security issues.

In [ ]:
# TODO: Write your system prompt
code_reviewer_prompt = """

"""

test_code = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return result
"""

response = llm.invoke([
    SystemMessage(content=code_reviewer_prompt),
    HumanMessage(content=f"Review this code:\n```python\n{test_code}\n```")
])
print(response.content)

<details>
<summary>💡 Solution</summary>

```python
code_reviewer_prompt = """You are a senior code reviewer.
- Find bugs, security vulnerabilities, and logic errors
- Ignore style/formatting unless it causes a bug
- Rate severity: 🔴 critical, 🟡 warning, 🟢 minor
- If the code is fine, say "LGTM"
"""
```
</details>

### Few-Shot Prompting

Show the model examples instead of describing what you want:

In [ ]:
# Few-shot: teach the model a specific output format
messages = [
    SystemMessage(content="Extract entities from text. Output as JSON."),
    HumanMessage(content="John works at Google in London."),
    AIMessage(content='{"people": ["John"], "organizations": ["Google"], "locations": ["London"]}'),
    HumanMessage(content="Tesla announced a new factory in Berlin with CEO Elon Musk."),
    AIMessage(content='{"people": ["Elon Musk"], "organizations": ["Tesla"], "locations": ["Berlin"]}'),
    HumanMessage(content="Microsoft's Satya Nadella visited the new campus in Redmond.")
]

response = llm.invoke(messages)
print(response.content)

### Prompt Templates and Chains

A **prompt template** lets you define reusable prompts with `{variable}` placeholders that get filled at runtime. **Chains** connect a template to a model using LangChain's pipe (`|`) operator — `template | llm` — so data flows from prompt construction to model invocation in a single expression. This is the building block for every pipeline in the rest of the course.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages([
    ("system", "You explain {topic} concepts to {audience}. Be concise."),
    ("human", "{question}")
])

# Chain: template → model → parse as string
chain = template | llm | StrOutputParser()

result = chain.invoke({
    "topic": "databases",
    "audience": "junior developers",
    "question": "When should I use NoSQL vs SQL?"
})
print(result)

### 🎯 Exercise: Design AskDoc's System Prompt

Create a system prompt that defines AskDoc's behavior: only answers from context, cites sources, says "I don't know" when unsure.

In [ ]:
# TODO: Design the system prompt
ASKDOC_SYSTEM_PROMPT = """

"""

askdoc_template = ChatPromptTemplate.from_messages([
    ("system", ASKDOC_SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

rag_chain = askdoc_template | llm | StrOutputParser()

# Test
print(rag_chain.invoke({
    "context": "The company was founded in 2020. It has 50 employees. The CEO is Jane Smith.",
    "question": "Who is the CEO?"
}))
print()
print(rag_chain.invoke({
    "context": "The company was founded in 2020.",
    "question": "What is the company's revenue?"
}))

<details>
<summary>💡 Solution</summary>

```python
ASKDOC_SYSTEM_PROMPT = """You are AskDoc, a document Q&A assistant.

Rules:
- Answer ONLY based on the provided context
- If the answer is not in the context, say: "I don't have enough information to answer that."
- Be concise: 1-3 sentences unless the user asks for detail
- Cite which part of the context your answer comes from
- Never make up information
"""
```
</details>

---
## Module 02: Embeddings & Semantic Search

An **embedding** is a fixed-length vector that encodes the *meaning* of text. Similar meanings map to nearby points in vector space. This is the foundation of all modern search and RAG systems.

Key concepts:
- **Cosine similarity** — measures the angle between vectors (1.0 = identical, 0 = unrelated)
- **Semantic vs keyword search** — "How can I get my money back?" matches "return policy" even with zero keyword overlap
- **Embedding dimensions** — typical models produce 256–3072 dimensional vectors

📖 Docs:
- [LangChain Embeddings](https://docs.langchain.com/oss/python/integrations/embeddings)
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)
- [What are Vector Embeddings? (Pinecone)](https://www.pinecone.io/learn/vector-embeddings/)

### Embedding Vectors & Cosine Similarity

Let's see embeddings in action. We'll embed three sentences and compare their vectors using **cosine similarity** — a value from 0 (unrelated) to 1 (identical meaning). The two cat/kitten sentences should score high; the stock market sentence should score low against both.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare semantically similar and different sentences
sentences = [
    "The cat sat on the mat",
    "A kitten was resting on the rug",
    "The stock market crashed yesterday",
]
vectors = [embeddings.embed_query(s) for s in sentences]

print(f"Base: '{sentences[0]}'\n")
for i in range(1, len(sentences)):
    sim = cosine_similarity(vectors[0], vectors[i])
    print(f"  vs '{sentences[i]}' → {sim:.4f}")

### Building Semantic Search

With embeddings and cosine similarity in hand, we can build a basic search engine. The idea: embed all your documents once, then at query time embed the user's question and find the closest document vectors. Notice how "How can I get my money back?" matches "return policy" — the model understands *meaning*, not just keywords.

In [ ]:
# Our knowledge base
documents = [
    "Our return policy allows returns within 30 days of purchase with a receipt.",
    "Free shipping is available on orders over  within the continental US.",
    "To reset your password, click 'Forgot Password' on the login page.",
    "Our customer support is available Monday to Friday, 9 AM to 5 PM EST.",
    "Premium members get 20% off all purchases and early access to new products.",
    "We accept Visa, MasterCard, American Express, and PayPal.",
    "Orders typically ship within 2 business days.",
    "Track your order using the tracking number in your confirmation email.",
]

doc_vectors = embeddings.embed_documents(documents)
print(f"Indexed {len(documents)} documents")

In [ ]:
def semantic_search(query: str, top_k: int = 3):
    """Find most relevant documents for a query."""
    query_vector = embeddings.embed_query(query)
    scores = [(cosine_similarity(query_vector, dv), documents[i]) for i, dv in enumerate(doc_vectors)]
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:top_k]

# "money back" matches "return policy" — no keyword overlap!
for score, doc in semantic_search("How can I get my money back?"):
    print(f"  [{score:.4f}] {doc}")

### Loading & Chunking Real Documents

In practice you don't embed hardcoded strings — you load files from disk (or URLs, databases, APIs). LangChain provides **document loaders** for many formats and **text splitters** to break large documents into embeddable chunks.

📖 Docs:
- [Document Loaders](https://docs.langchain.com/oss/python/integrations/document_loaders)
- [Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- [TextLoader](https://docs.langchain.com/oss/python/integrations/document_loaders)
- [DirectoryLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/directory)

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load a single file
loader = TextLoader("../data/handbook.md")
docs = loader.load()
print(f"Loaded {len(docs)} document(s) from handbook.md")
print(f"  Characters: {len(docs[0].page_content)}")
print(f"  Metadata: {docs[0].metadata}")

In [ ]:
# Load ALL files from a directory (supports glob patterns)
dir_loader = DirectoryLoader("../data/", glob="**/*.md", loader_cls=TextLoader)
all_docs = dir_loader.load()
print(f"Loaded {len(all_docs)} document(s) from data/")
for doc in all_docs:
    print(f"  {doc.metadata['source']} — {len(doc.page_content)} chars")

# Also load .txt files
txt_loader = DirectoryLoader("../data/", glob="**/*.txt", loader_cls=TextLoader)
txt_docs = txt_loader.load()
all_docs.extend(txt_docs)
print(f"\nTotal documents: {len(all_docs)}")

In [ ]:
# Chunk documents for embedding
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)
print(f"Split {len(all_docs)} documents into {len(chunks)} chunks\n")

# Inspect a few chunks
for i, chunk in enumerate(chunks[:3]):
    source = chunk.metadata.get("source", "?")
    print(f"Chunk {i} ({source}):")
    print(f"  {chunk.page_content[:100]}...")
    print()

In [ ]:
# Embed chunks and search across all loaded documents
from langchain_community.vectorstores import Chroma

file_vectorstore = Chroma.from_documents(chunks, embeddings)
results = file_vectorstore.similarity_search_with_score("How do code reviews work?", k=3)

print("Search results across all files:\n")
for doc, score in results:
    source = doc.metadata.get("source", "?")
    print(f"  [{score:.4f}] ({source})")
    print(f"  {doc.page_content[:120]}...")
    print()

### 🎯 Exercise: Build a DocSearcher Class

Create a reusable search engine class for AskDoc.

In [ ]:
class DocSearcher:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.documents = []
        self.vectors = []

    def add_documents(self, docs: list[str]):
        # TODO: Store docs and compute embeddings
        pass

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        # TODO: Return [{"text": ..., "score": ...}, ...]
        pass

# Test
searcher = DocSearcher(embeddings)
searcher.add_documents(documents)
for r in searcher.search("How do I track my package?"):
    print(f"  [{r['score']:.4f}] {r['text']}")

<details>
<summary>💡 Solution</summary>

```python
class DocSearcher:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.documents = []
        self.vectors = []

    def add_documents(self, docs: list[str]):
        self.documents.extend(docs)
        self.vectors.extend(self.embedding_model.embed_documents(docs))

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        query_vector = self.embedding_model.embed_query(query)
        scores = []
        for i, doc_vec in enumerate(self.vectors):
            sim = cosine_similarity(query_vector, doc_vec)
            scores.append({"text": self.documents[i], "score": sim})
        scores.sort(key=lambda x: x["score"], reverse=True)
        return scores[:top_k]
```
</details>

---
## Module 03: RAG Fundamentals

**Retrieval-Augmented Generation (RAG)** is the most common production AI pattern. Instead of relying on the model's training data, you retrieve relevant documents and inject them as context.

The pipeline: `User Query → Embed → Search Chunks → Build Prompt with Context → LLM → Answer`

Key decisions:
- **Chunk size** — too small loses context, too large dilutes relevance (sweet spot: 500-1000 chars)
- **Overlap** — adjacent chunks share text so sentences aren't cut mid-thought
- **Vector store** — ChromaDB for prototyping, FAISS/Pinecone for production scale

📖 Docs:
- [LangChain RAG Tutorial](https://docs.langchain.com/oss/python/langchain/rag)
- [Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- [Vector Stores](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [ChromaDB Getting Started](https://docs.trychroma.com/docs/overview/getting-started)

### Load, Chunk, and Index

RAG starts by loading documents from disk, splitting them into chunks small enough to embed, and storing the chunks in a vector database. Below we use `DirectoryLoader` to load all files from `data/`, `RecursiveCharacterTextSplitter` to break them into ~500-char chunks with overlap, and ChromaDB to index them.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_core.documents import Document

# Load all documents from the data/ directory
md_loader = DirectoryLoader("../data/", glob="**/*.md", loader_cls=TextLoader)
txt_loader = DirectoryLoader("../data/", glob="**/*.txt", loader_cls=TextLoader)
raw_docs = md_loader.load() + txt_loader.load()

print(f"Loaded {len(raw_docs)} documents from data/:")
for doc in raw_docs:
    print(f"  {doc.metadata['source']} — {len(doc.page_content)} chars")

In [ ]:
# Chunk all loaded documents
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(raw_docs)
print(f"Split {len(raw_docs)} documents into {len(chunks)} chunks")

# Store in ChromaDB
vectorstore = Chroma.from_documents(chunks, embeddings)
print(f"Indexed {len(chunks)} chunks in ChromaDB")

### Query the RAG Pipeline

Now the payoff: ask a question in natural language, retrieve the most relevant chunks, inject them as context into the prompt, and let the LLM answer *only from what it was given*. Notice the second question — "stock price" — gets an honest "I don't know" because nothing in our documents covers it.

In [ ]:
# Full RAG pipeline
def ask(question: str) -> str:
    results = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join(r.page_content for r in results)
    return rag_chain.invoke({"context": context, "question": question})

print(ask("How many vacation days do I get?"))
print()
print(ask("What is the company's stock price?"))

### 🎯 Exercise: Build the Full AskDoc RAG Class

In [ ]:
class AskDoc:
    """RAG-powered document Q&A assistant."""

    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.vectorstore = None
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        self.chain = self._build_chain()

    def _build_chain(self):
        # TODO: Create RAG prompt template + chain
        pass

    def ingest_file(self, path: str):
        # TODO: Load file from disk, chunk, add to vectorstore
        pass

    def ingest_directory(self, path: str, glob: str = "**/*.*"):
        # TODO: Load all matching files from a directory
        pass

    def ask(self, question: str) -> str:
        # TODO: Retrieve + generate
        pass

# Test
askdoc = AskDoc(llm, embeddings)
askdoc.ingest_directory("../data/", glob="**/*.md")
print(askdoc.ask("How do performance reviews work?"))

<details>
<summary>💡 Solution</summary>

```python
class AskDoc:
    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.vectorstore = None
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        self.chain = self._build_chain()

    def _build_chain(self):
        template = ChatPromptTemplate.from_messages([
            ("system", """You are AskDoc. Answer ONLY from the provided context.
If unsure, say so. Be concise."""),
            ("human", "Context:\n{context}\n\nQuestion: {question}")
        ])
        return template | self.llm | StrOutputParser()

    def ingest_file(self, path: str):
        loader = TextLoader(path)
        docs = loader.load()
        chunks = self.splitter.split_documents(docs)
        if self.vectorstore is None:
            self.vectorstore = Chroma.from_documents(chunks, self.embeddings)
        else:
            self.vectorstore.add_documents(chunks)

    def ingest_directory(self, path: str, glob: str = "**/*.*"):
        loader = DirectoryLoader(path, glob=glob, loader_cls=TextLoader)
        docs = loader.load()
        chunks = self.splitter.split_documents(docs)
        if self.vectorstore is None:
            self.vectorstore = Chroma.from_documents(chunks, self.embeddings)
        else:
            self.vectorstore.add_documents(chunks)

    def ask(self, question: str) -> str:
        if not self.vectorstore:
            return "No documents ingested yet."
        docs = self.vectorstore.similarity_search(question, k=3)
        context = "\n\n".join(d.page_content for d in docs)
        return self.chain.invoke({"context": context, "question": question})
```
</details>

---
## Module 04: Vector Databases & Metadata Filtering

In-memory search doesn't scale. **Vector databases** persist embeddings to disk and support fast approximate nearest-neighbor (ANN) search over millions of documents.

Key concepts:
- **FAISS** — Facebook's library for efficient similarity search; great for local/offline use
- **ChromaDB** — developer-friendly vector DB with built-in metadata filtering and persistence
- **Metadata filtering** — attach key-value pairs to documents (e.g., `source`, `date`, `category`) and filter at query time to narrow results before similarity ranking
- **Hybrid search** — combine semantic similarity with keyword matching for better recall

📖 Docs:
- [ChromaDB Docs](https://docs.trychroma.com/)
- [FAISS Wiki](https://github.com/facebookresearch/faiss/wiki)
- [LangChain Vector Stores](https://docs.langchain.com/oss/python/integrations/vectorstores)
- [LangChain Retrievers](https://docs.langchain.com/oss/python/integrations/retrievers)

### Metadata Filtering with ChromaDB

Attach key-value **metadata** to each document (department, type, date, etc.) and filter at query time. This narrows the search space *before* similarity ranking — "show me only engineering policies" — which improves both relevance and speed when you have documents from many sources.

In [ ]:
# Documents from different departments
all_docs = [
    Document(page_content="Annual leave is 25 days for full-time employees.", metadata={"dept": "hr", "type": "policy"}),
    Document(page_content="Parental leave is 16 weeks fully paid.", metadata={"dept": "hr", "type": "policy"}),
    Document(page_content="All code must pass CI before merging to main.", metadata={"dept": "engineering", "type": "process"}),
    Document(page_content="Use Python 3.11+ for all new services.", metadata={"dept": "engineering", "type": "standard"}),
    Document(page_content="Expenses over  require manager approval.", metadata={"dept": "finance", "type": "policy"}),
    Document(page_content="Invoices are processed on the 1st and 15th.", metadata={"dept": "finance", "type": "process"}),
]

store = Chroma.from_documents(all_docs, embeddings, collection_name="company_docs")
print(f"Indexed {store._collection.count()} documents with metadata")

In [ ]:
# Without filter
print("All departments:")
for r in store.similarity_search("What are the rules?", k=3):
    print(f"  [{r.metadata['dept']}] {r.page_content}")

print()

# With metadata filter
print("Engineering only:")
for r in store.similarity_search("What are the rules?", k=3, filter={"dept": "engineering"}):
    print(f"  [{r.metadata['dept']}] {r.page_content}")

### FAISS — High-Performance Alternative

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(all_docs, embeddings)
results = faiss_store.similarity_search_with_score("time off", k=2)
for doc, score in results:
    print(f"  [{score:.4f}] {doc.page_content}")

faiss_store.save_local("../data/faiss_index")
print("\n✅ FAISS index saved to disk")

---
## Module 05: Tool & Function Calling

LLMs can only generate text — **tools** give them the ability to act on the world. When the model decides it needs external information or needs to perform an action, it emits a structured tool call instead of a text response.

How it works:
1. You describe available tools (name, description, parameters) to the model
2. The model decides *when* to call a tool and *what arguments* to pass
3. Your code executes the tool and returns the result to the model
4. The model incorporates the result into its final answer

Common tool patterns: web search, database queries, calculations, API calls, file operations.

📖 Docs:
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [LangChain Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)

### Defining Tools

A **tool** is a Python function decorated with `@tool`. The decorator extracts the function's name, docstring, and type hints to generate a schema that the LLM can understand. When the model wants to use a tool, it returns a structured JSON call with the function name and arguments — your code then executes it.

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression safely.

    Args:
        expression: Math like '25 - (3 * 7)'
    """
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "Error: Only basic math allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get current date and time.

    Args:
        timezone: e.g., 'UTC', 'US/Eastern'
    """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S") + f" ({timezone})"

print(f"Tools defined: {calculate.name}, {get_current_time.name}")

### The Tool-Calling Loop

Calling a tool is a multi-step conversation: (1) send the user's question + tool descriptions to the LLM, (2) the LLM responds with a `tool_calls` list instead of text, (3) you execute each tool and send results back as `ToolMessage`s, (4) the LLM incorporates the results into a final text answer. This loop may repeat if the model needs multiple tools.

In [ ]:
# Bind tools to LLM
tools = [calculate, get_current_time]
llm_with_tools = llm.bind_tools(tools)

# LLM decides to use a tool
response = llm_with_tools.invoke("What is 1547 * 23?")
print(f"Tool calls: {response.tool_calls}")

In [ ]:
def run_with_tools(user_input: str, tools: list, llm) -> str:
    """Complete tool-calling loop."""
    tool_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)
    messages = [HumanMessage(content=user_input)]

    for _ in range(5):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for tc in response.tool_calls:
            result = tool_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return "Max iterations reached."

print(run_with_tools("What is (15 + 7) * 3?", tools, llm))
print()
print(run_with_tools("What time is it?", tools, llm))

### 🎯 Exercise: Add a Document Search Tool

Create a `search_documents` tool that the LLM can call.

In [ ]:
@tool
def search_documents(query: str) -> str:
    """Search company documents for information.

    Args:
        query: The search query
    """
    # TODO: Use the vectorstore from Module 03 to search
    # Hint: vectorstore.similarity_search(query, k=2)
    pass

all_tools = [calculate, get_current_time, search_documents]
# print(run_with_tools("What is the leave policy?", all_tools, llm))

<details>
<summary>💡 Solution</summary>

```python
@tool
def search_documents(query: str) -> str:
    """Search company documents for information.

    Args:
        query: The search query
    """
    if not vectorstore:
        return "No documents indexed."
    results = vectorstore.similarity_search(query, k=2)
    return "
".join(r.page_content for r in results)
```
</details>

---
## Module 06: Agentic Workflows with LangGraph

An **agent** is an LLM that can decide its own control flow — choosing which tools to call, when to loop, and when to stop. **LangGraph** models this as a state machine (graph) where:

- **Nodes** are functions that transform state (call LLM, run tool, check condition)
- **Edges** define transitions between nodes (including conditional routing)
- **State** is a typed dict that accumulates information across steps

This is more powerful than simple chains because the agent can reason about intermediate results and adapt its plan. The graph structure also makes complex flows debuggable and testable.

📖 Docs:
- [LangGraph Docs](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangGraph Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [ReAct Agent Pattern](https://docs.langchain.com/oss/python/langchain/agents)
- [LangGraph Concepts](https://docs.langchain.com/oss/python/langgraph/overview)

### Building a ReAct Agent Graph

Below we define a **state graph** with two nodes: `agent` (calls the LLM) and `tools` (executes tool calls). A conditional edge checks whether the LLM's response contains tool calls — if yes, route to the tools node and loop back; if no, the agent is done. This is the **ReAct** (Reason + Act) pattern: the LLM reasons about what to do, acts via tools, observes results, and repeats until it has a final answer.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# Agent node: LLM decides what to do
def agent_node(state: AgentState) -> AgentState:
    messages = [SystemMessage(content="You are AskDoc. Use tools to find info. Be concise.")] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}

def should_continue(state: AgentState) -> str:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

# Build the graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(all_tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, ["tools", END])
graph.add_edge("tools", "agent")

agent = graph.compile()
print("Agent graph: START → agent → (tools → agent)* → END")

In [ ]:
# Test the agent
result = agent.invoke({"messages": [HumanMessage(content="What is 25 - 3 * 7?")]})
print(f"A: {result['messages'][-1].content}")

### 🎯 Exercise: Build AskDocAgent Class

In [ ]:
class AskDocAgent:
    def __init__(self, llm, tools: list):
        self.llm = llm
        self.tools = tools
        self.graph = self._build_graph()

    def _build_graph(self):
        # TODO: Build a StateGraph with agent + tools nodes
        pass

    def ask(self, question: str) -> str:
        # TODO: Invoke graph and return final message
        pass

# askdoc_agent = AskDocAgent(llm, all_tools)
# print(askdoc_agent.ask("What is the remote work policy?"))

<details>
<summary>💡 Solution</summary>

```python
class AskDocAgent:
    def __init__(self, llm, tools: list):
        self.llm = llm
        self.tools = tools
        self.graph = self._build_graph()

    def _build_graph(self):
        llm_with_tools = self.llm.bind_tools(self.tools)

        def agent_node(state: AgentState) -> AgentState:
            msgs = [SystemMessage(content="You are AskDoc. Use tools when needed.")] + state["messages"]
            return {"messages": [llm_with_tools.invoke(msgs)]}

        def should_continue(state: AgentState) -> str:
            if state["messages"][-1].tool_calls:
                return "tools"
            return END

        g = StateGraph(AgentState)
        g.add_node("agent", agent_node)
        g.add_node("tools", ToolNode(self.tools))
        g.add_edge(START, "agent")
        g.add_conditional_edges("agent", should_continue, ["tools", END])
        g.add_edge("tools", "agent")
        return g.compile()

    def ask(self, question: str) -> str:
        result = self.graph.invoke({"messages": [HumanMessage(content=question)]})
        return result["messages"][-1].content
```
</details>

---
## Module 07: Structured Generation

Free-text output is hard to parse reliably. **Structured generation** forces the model to emit JSON conforming to a schema — no regex parsing, no "please format as JSON" prayers.

LangChain's `with_structured_output()` uses the model's native function-calling or JSON mode to guarantee valid output. Combined with Pydantic models, you get:
- Type-safe responses with automatic validation
- Nested objects, lists, enums, and optional fields
- Automatic retry on validation failure

This is essential for building reliable pipelines where downstream code expects specific data shapes.

📖 Docs:
- [LangChain Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [Pydantic Models](https://docs.pydantic.dev/latest/concepts/models/)
- [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)

### Using `with_structured_output()`

Define a **Pydantic model** that describes the shape of the response you want, then pass it to `llm.with_structured_output(YourModel)`. The LLM is now constrained to return a valid instance of that model — no more regex parsing or hoping the JSON is well-formed. The result is a Python object with typed attributes you can access directly.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PersonInfo(BaseModel):
    name: str = Field(description="Full name")
    age: int = Field(description="Age")
    city: str = Field(description="City")

structured_llm = llm.with_structured_output(PersonInfo)
result = structured_llm.invoke("John is 30 and lives in London.")
print(f"Name: {result.name}, Age: {result.age}, City: {result.city}")

### 🎯 Exercise: Structured AskDoc Responses

In [ ]:
class AskDocResponse(BaseModel):
    """Structured response from AskDoc."""
    answer: str = Field(description="The answer")
    confidence: float = Field(description="0.0 to 1.0")
    sources: list[str] = Field(description="Document sections used")
    follow_up_questions: list[str] = Field(description="2-3 suggested follow-ups")

# TODO: Use llm.with_structured_output(AskDocResponse)
# to get structured answers from AskDoc

# context = "Annual leave is 25 days. Remote work 3 days/week."
# askdoc_structured = llm.with_structured_output(AskDocResponse)
# result = askdoc_structured.invoke(f"Context: {context}\nQuestion: How many days can I work from home?")
# print(result.model_dump_json(indent=2))

<details>
<summary>💡 Solution</summary>

```python
context = "Annual leave is 25 days. Remote work 3 days/week."
askdoc_structured = llm.with_structured_output(AskDocResponse)
result = askdoc_structured.invoke(f"Context: {context}
Question: How many days can I work from home?")
print(result.model_dump_json(indent=2))
```
</details>

---
## Module 08: Memory & Conversation

Without memory, every request is independent — the model has no idea what you said 30 seconds ago. **Conversation memory** solves this by injecting prior messages into the prompt.

Strategies:
- **Buffer memory** — keep all messages (simple, but hits token limits fast)
- **Window memory** — keep only the last N exchanges
- **Summary memory** — periodically summarize older messages to compress history
- **LangGraph state** — store messages in graph state for agent workflows

In LangGraph, memory is just part of the state dict. You define a `messages` key and the framework handles appending new messages each turn.

📖 Docs:
- [LangGraph Memory](https://docs.langchain.com/oss/python/langgraph/overviewmemory/)
- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/overviewpersistence/)
- [LangChain Message History](https://docs.langchain.com/oss/python/langchain/short-term-memory)

### LLMs Are Stateless — Proof

Each `llm.invoke()` call is completely independent. The model has no built-in memory of previous turns. We'll prove this by telling the model our name in one call, then asking it in a separate call — it won't know.

In [ ]:
# LLMs are stateless — prove it
r1 = llm.invoke([HumanMessage(content="My name is Ahmed.")])
r2 = llm.invoke([HumanMessage(content="What is my name?")])
print(f"Turn 1: {r1.content}")
print(f"Turn 2: {r2.content}  ← doesn't remember!")

### Adding Memory with `MemorySaver`

LangGraph solves the statelessness problem with **checkpointers**. `MemorySaver` stores the full message history for each `thread_id`. When you invoke the graph with the same thread, previous messages are automatically included — the model "remembers." A different thread starts fresh.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# LangGraph with built-in memory
def chat_node(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content="You are AskDoc. Be concise.")] + state["messages"]
    return {"messages": [llm.invoke(msgs)]}

g = StateGraph(AgentState)
g.add_node("chat", chat_node)
g.add_edge(START, "chat")
g.add_edge("chat", END)

memory_app = g.compile(checkpointer=MemorySaver())

# Same thread = remembers
config = {"configurable": {"thread_id": "user-123"}}
r1 = memory_app.invoke({"messages": [HumanMessage(content="My name is Ahmed.")]}, config)
print(f"Turn 1: {r1['messages'][-1].content}")

r2 = memory_app.invoke({"messages": [HumanMessage(content="What's my name?")]}, config)
print(f"Turn 2: {r2['messages'][-1].content}  ← remembers!")

# Different thread = fresh conversation
config2 = {"configurable": {"thread_id": "user-456"}}
r3 = memory_app.invoke({"messages": [HumanMessage(content="What's my name?")]}, config2)
print(f"\nNew thread: {r3['messages'][-1].content}")

### 🎯 Exercise: AskDoc Agent with Memory

Combine the agent from Module 06 with LangGraph's MemorySaver.

In [ ]:
# TODO: Build agent graph with checkpointer=MemorySaver()
# Test: ask about leave, then remote work, then "summarize what you told me"

# memory_agent = ...
# config = {"configurable": {"thread_id": "ahmed"}}
# print(memory_agent.invoke({"messages": [HumanMessage(content="What's the leave policy?")]}, config))
# print(memory_agent.invoke({"messages": [HumanMessage(content="And remote work?")]}, config))
# print(memory_agent.invoke({"messages": [HumanMessage(content="Summarize both")]}, config))

<details>
<summary>💡 Solution</summary>

```python
def agent_with_mem(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content="You are AskDoc. Use tools when needed.")] + state["messages"]
    return {"messages": [llm_with_tools.invoke(msgs)]}

def should_cont(state: AgentState) -> str:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

g = StateGraph(AgentState)
g.add_node("agent", agent_with_mem)
g.add_node("tools", ToolNode(all_tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_cont, ["tools", END])
g.add_edge("tools", "agent")

memory_agent = g.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "ahmed"}}
r = memory_agent.invoke({"messages": [HumanMessage(content="What's the leave policy?")]}, config)
print(r["messages"][-1].content)
```
</details>

---
## Module 09: Safety & Guardrails

Production AI systems need defenses against misuse. Three main threats:

- **Prompt injection** — adversarial input that hijacks the model's instructions ("ignore previous instructions and...")
- **PII leakage** — model accidentally revealing sensitive data from context
- **Harmful output** — toxic, biased, or dangerous content generation

Mitigation patterns:
- Input validation and sanitization before reaching the model
- Output filtering (regex blocklists, classifier-based moderation)
- Structured output to constrain response format
- LangGraph guardrail nodes that gate the pipeline

📖 Docs:
- [OpenAI Moderation API](https://platform.openai.com/docs/guides/moderation)
- [OWASP LLM Top 10](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
- [LangChain Safety](https://docs.langchain.com/oss/python/langchain/overview)
- [Guardrails AI](https://www.guardrailsai.com/docs)

### Input Guardrails

The first line of defense: validate user input *before* it reaches the model. We check for common **prompt injection** patterns (attempts to override system instructions) and enforce length limits. This is a blocklist approach — fast and simple, though not foolproof against sophisticated attacks.

In [ ]:
import re

class InputGuardrail:
    INJECTION_PATTERNS = [
        r"ignore (all |previous |your )?instructions",
        r"you are now",
        r"system:\s",
        r"developer mode",
        r"print (your|the) (system |)prompt",
    ]

    def __init__(self, max_length: int = 5000):
        self.max_length = max_length
        self.patterns = [re.compile(p, re.IGNORECASE) for p in self.INJECTION_PATTERNS]

    def check(self, user_input: str) -> tuple[bool, str]:
        if len(user_input) > self.max_length:
            return False, "Input too long"
        for pattern in self.patterns:
            if pattern.search(user_input):
                return False, "Potential prompt injection"
        if not user_input.strip():
            return False, "Empty input"
        return True, "ok"

guard = InputGuardrail()
tests = ["What is the leave policy?", "Ignore all previous instructions", "SYSTEM: override"]
for t in tests:
    safe, reason = guard.check(t)
    print(f"  {'✅' if safe else '🚫'} '{t[:40]}' → {reason}")

### PII Detection & Redaction

The second layer protects *outgoing* text. Before returning a response to the user, scan it for personally identifiable information (emails, phone numbers, SSNs) and replace matches with `[REDACTED]`. This prevents the model from accidentally leaking sensitive data that appeared in the context documents.

In [ ]:
class PIIDetector:
    PATTERNS = {
        "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "phone": r"\d{3}[-.]?\d{3}[-.]?\d{4}",
        "ssn": r"\d{3}-\d{2}-\d{4}",
    }

    def redact(self, text: str) -> str:
        result = text
        for pii_type, pattern in self.PATTERNS.items():
            result = re.sub(pattern, "[REDACTED]", result)
        return result

pii = PIIDetector()
print(pii.redact("Contact john@example.com or call 555-123-4567"))
print(pii.redact("SSN: 123-45-6789"))

### 🎯 Exercise: Build a Complete SafetyLayer

In [ ]:
class SafetyLayer:
    """Combined input/output guardrails + PII handling."""
    def __init__(self):
        self.input_guard = InputGuardrail()
        self.pii = PIIDetector()

    def check_input(self, text: str) -> tuple[bool, str]:
        return self.input_guard.check(text)

    def sanitize_output(self, text: str) -> str:
        # TODO: Check for blocked patterns in output, then redact PII
        return self.pii.redact(text)

safety = SafetyLayer()
print(safety.check_input("Normal question"))
print(safety.sanitize_output("Email me at test@example.com"))

---
## Module 10: Human-in-the-Loop

Not every action should be automated. **Human-in-the-loop (HITL)** adds approval gates where the AI proposes an action and a human confirms or rejects it before execution.

Use cases:
- High-risk operations (deleting data, sending emails, financial transactions)
- Low-confidence decisions where the model is uncertain
- Compliance requirements that mandate human oversight

In LangGraph, HITL is implemented with **interrupts** — the graph pauses at a node, surfaces the pending action to the user, and resumes only after approval. The `interrupt()` function makes this trivial.

📖 Docs:
- [LangGraph Human-in-the-Loop](https://docs.langchain.com/oss/python/langgraph/overviewhuman_in_the_loop/)
- [LangGraph Breakpoints](https://docs.langchain.com/oss/python/langgraph/human-in-the-loop)
- [LangGraph interrupt()](https://docs.langchain.com/oss/python/langgraph/human-in-the-loop)

### Classifying Tools by Risk Level

Not all tools are equal. Read-only tools like `search_documents` are safe to auto-execute, but tools that send emails or submit requests need human approval. We split tools into **safe** (auto-execute) and **dangerous** (require confirmation) sets, then gate the dangerous ones behind an approval step.

In [ ]:
# Classify tools by risk
SAFE_TOOLS = {"search_documents", "calculate", "get_current_time"}
DANGEROUS_TOOLS = {"send_email", "delete_document", "submit_request"}

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email (requires approval).

    Args:
        to: Recipient email
        subject: Subject line
        body: Email body
    """
    return f"✅ Email sent to {to}: '{subject}'"

@tool
def submit_request(request_type: str, details: str) -> str:
    """Submit an HR request (requires approval).

    Args:
        request_type: Type of request (leave, expense, etc.)
        details: Request details
    """
    return f"✅ {request_type} request submitted: {details}"

print("Defined high-risk tools: send_email, submit_request")

### The Approval Loop

The approval function intercepts dangerous tool calls before execution. In production this would send a notification to Slack, a queue, or a UI — here we auto-approve for demonstration. If the human rejects the action, we return a denial message to the LLM so it can adjust its response.

In [ ]:
def human_approval(tool_name: str, args: dict) -> bool:
    """Simulate human approval for dangerous actions."""
    print(f"\n⚠️  APPROVAL NEEDED")
    print(f"   Tool: {tool_name}")
    print(f"   Args: {json.dumps(args, indent=2)}")
    # In production: send to a queue, Slack, email, etc.
    # Here we auto-approve for demo
    print(f"   → Auto-approved (demo mode)")
    return True

def run_with_approval(user_input: str, tools: list, llm) -> str:
    """Tool loop with human approval for dangerous tools."""
    tool_map = {t.name: t for t in tools}
    llm_bound = llm.bind_tools(tools)
    messages = [HumanMessage(content=user_input)]

    for _ in range(5):
        response = llm_bound.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for tc in response.tool_calls:
            if tc["name"] in DANGEROUS_TOOLS:
                if not human_approval(tc["name"], tc["args"]):
                    messages.append(ToolMessage(content="Action denied by user.", tool_call_id=tc["id"]))
                    continue
            result = tool_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return "Max iterations."

hitl_tools = [calculate, get_current_time, search_documents, send_email, submit_request]
print(run_with_approval("Send an email to bob@co.com saying hi", hitl_tools, llm))

---
## Module 11: Capstone — Complete AskDoc

Put it all together into a production-grade document assistant. This capstone combines every module:

| Capability | Module |
|---|---|
| Document retrieval | RAG (03) + Vector DBs (04) |
| External actions | Tools (05) |
| Adaptive reasoning | Agents (06) |
| Typed responses | Structured Output (07) |
| Context across turns | Memory (08) |
| Input/output protection | Safety (09) |
| Approval gates | HITL (10) |

The final AskDoc is a LangGraph agent with a RAG retriever tool, conversation memory, structured output, guardrail nodes, and human approval for sensitive operations.

📖 Docs:
- [LangGraph End-to-End Tutorial](https://docs.langchain.com/oss/python/langgraph/quickstart)
- [LangChain Use Cases](https://docs.langchain.com/oss/python/langchain/overview)
- [Building Production RAG (LangChain Blog)](https://blog.langchain.dev/)

In [ ]:
# The complete AskDoc system — fill in the pieces from previous modules

class FinalAskDoc:
    """Production AskDoc: RAG + Agent + Memory + Safety + HITL."""

    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.safety = SafetyLayer()
        self.vectorstore = None
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        self.app = self._build_agent()

    def _build_agent(self):
        # TODO: Combine everything:
        # 1. Define tools (search_docs using self.vectorstore, calculate, get_current_time)
        # 2. Build LangGraph agent with MemorySaver
        # 3. Add safety checks in the flow
        pass

    def ingest(self, text: str, source: str):
        """Add a document to the knowledge base."""
        chunks = self.splitter.split_text(text)
        docs = [Document(page_content=c, metadata={"source": source}) for c in chunks]
        if self.vectorstore is None:
            self.vectorstore = Chroma.from_documents(docs, self.embeddings)
        else:
            self.vectorstore.add_documents(docs)

    def chat(self, question: str, user_id: str = "default") -> str:
        """Full pipeline: safety check → agent → safety check → response."""
        # Input safety
        is_safe, reason = self.safety.check_input(question)
        if not is_safe:
            return f"⚠️ Blocked: {reason}"

        # TODO: Invoke agent with memory (use user_id as thread_id)
        # TODO: Sanitize output before returning
        return "Not implemented yet"


# When you're ready:
# final = FinalAskDoc(llm, embeddings)
# final.ingest(sample_doc, "handbook")
# print(final.chat("What's the leave policy?", user_id="ahmed"))
# print(final.chat("And remote work?", user_id="ahmed"))
# print(final.chat("Summarize both", user_id="ahmed"))

<details>
<summary>💡 Complete Solution</summary>

```python
class FinalAskDoc:
    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.safety = SafetyLayer()
        self.vectorstore = None
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        self._tools = None
        self._app = None

    def _get_tools(self):
        if self._tools:
            return self._tools
        vs = self.vectorstore

        @tool
        def search_docs(query: str) -> str:
            """Search company documents.
            Args:
                query: Search query
            """
            if not vs:
                return "No documents indexed."
            results = vs.similarity_search(query, k=3)
            return "
".join(r.page_content for r in results)

        self._tools = [search_docs, calculate, get_current_time]
        return self._tools

    def _build_agent(self):
        tools = self._get_tools()
        llm_with_tools = self.llm.bind_tools(tools)

        def agent_node(state):
            msgs = [SystemMessage(content="You are AskDoc. Use tools to answer from documents.")] + state["messages"]
            return {"messages": [llm_with_tools.invoke(msgs)]}

        def should_continue(state):
            if state["messages"][-1].tool_calls:
                return "tools"
            return END

        g = StateGraph(AgentState)
        g.add_node("agent", agent_node)
        g.add_node("tools", ToolNode(tools))
        g.add_edge(START, "agent")
        g.add_conditional_edges("agent", should_continue, ["tools", END])
        g.add_edge("tools", "agent")
        return g.compile(checkpointer=MemorySaver())

    def ingest(self, text: str, source: str):
        chunks = self.splitter.split_text(text)
        docs = [Document(page_content=c, metadata={"source": source}) for c in chunks]
        if self.vectorstore is None:
            self.vectorstore = Chroma.from_documents(docs, self.embeddings)
        else:
            self.vectorstore.add_documents(docs)
        self._tools = None
        self._app = None

    def chat(self, question: str, user_id: str = "default") -> str:
        is_safe, reason = self.safety.check_input(question)
        if not is_safe:
            return f"⚠️ Blocked: {reason}"

        if not self._app:
            self._app = self._build_agent()

        config = {"configurable": {"thread_id": user_id}}
        result = self._app.invoke({"messages": [HumanMessage(content=question)]}, config)
        output = result["messages"][-1].content
        return self.safety.sanitize_output(output)
```
</details>

---
## 🎉 Course Complete!

You've built a complete AI application from scratch:

| Module | What You Built |
|--------|---------------|
| 00 | LLM provider factory |
| 01 | System prompts, templates, chains |
| 02 | Semantic search with embeddings |
| 03 | Full RAG pipeline |
| 04 | Vector databases with filtering |
| 05 | Tool calling |
| 06 | LangGraph agents |
| 07 | Structured output |
| 08 | Conversation memory |
| 09 | Safety guardrails |
| 10 | Human-in-the-loop |
| 11 | Everything combined |

**AskDoc** is now a production-ready AI assistant with RAG, multi-step reasoning, memory, safety, and human oversight.